# 3 — Construire un mélange gaussien : fonction, classe, module

**Atelier repris du 22 septembre et intégré au cours du 24 septembre.**
Vous écrivez les lignes marquées `TODO`
dans le notebook, vérifiez chaque étape, puis créez vous-même un
fichier `my_gaussian_mixture.py` dans l'éditeur. Le notebook ne crée
ni ne réécrit ce fichier.

Une classe n'est pas requise pour une simulation unique. Ici elle
devient utile parce que plusieurs opérations partagent les mêmes
paramètres et le même échantillon : produire les données, afficher
les composantes, le mélange et une comparaison par boîte à moustaches.

**Dans cette simulation :** chaque observation vient de A **ou** de B.
On n'additionne pas deux valeurs. Le code fixe le nombre de lignes de
A à `int(n * weight_a)` et attribue les lignes restantes à B. Pour
`n=2000` et `weight_a=0.6`, cela donne 1 200 lignes de A et 800 de B.
Avec un très petit effectif ou un poids proche de zéro, ce calcul peut
produire un groupe vide. Pour pouvoir comparer les deux groupes dans
les graphiques, votre méthode `sample` devra refuser ce cas : il faut
au moins une observation de A et une de B. Cette simulation sert à
organiser le code ; elle ne prouve pas l'origine de données observées.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

print("Current directory:", Path.cwd())

## 1. Lire une simulation directe

Deux distributions normales sont utilisées : A a une moyenne de 100
et un écart-type de 18 ; B a une moyenne de 106 et un écart-type de 24.
Leurs distributions se recouvrent ; une valeur isolée ne révèle pas
sûrement sa composante. `seed` fixe la séquence pseudo-aléatoire
pour rendre l'exemple reproductible.

**Avant d'exécuter :** prédisez le nombre de lignes de chaque groupe
et le type de `values_a`.

In [ ]:
n = 2000
weight_a = 0.6
n_a = int(n * weight_a)
n_b = n - n_a
rng = np.random.default_rng(404)
values_a = rng.normal(loc=100, scale=18, size=n_a)
values_b = rng.normal(loc=106, scale=24, size=n_b)
baseline = pd.DataFrame({
    "value": np.concatenate([values_a, values_b]),
    "component": ["A"] * n_a + ["B"] * n_b,
})
print(type(values_a).__name__, values_a.shape, baseline.shape)
display(baseline.groupby("component")["value"].agg(["count", "mean", "std"]))

## 2. Extraire une fonction

Complétez `draw_component`. `rng.normal(loc=..., scale=..., size=...)`
renvoie un tableau NumPy. La fonction reçoit le générateur pour que
l'appelant décide de la graine ; elle **renvoie** le tableau au lieu
de l'afficher. Une définition `def` n'exécute pas son corps.

In [ ]:
def draw_component(rng, mean, std_dev, size):
    """Draw size values from one Gaussian distribution."""
    # TODO: return the NumPy array produced by rng.normal(...).
    return None

In [ ]:
candidate = draw_component(np.random.default_rng(7), 100, 18, 5)
if candidate is None:
    print("À compléter : draw_component doit renvoyer un tableau de 5 valeurs.")
else:
    assert isinstance(candidate, np.ndarray) and candidate.shape == (5,)
    expected = np.random.default_rng(7).normal(loc=100, scale=18, size=5)
    np.testing.assert_array_equal(candidate, expected)
    print("Fonction validée :", candidate.round(1))

## 3. Donner une structure aux résultats

Complétez `combine_components`. La sortie attendue est une DataFrame
avec une observation par ligne et les colonnes `value` et `component`. `np.concatenate`
assemble les valeurs, la répétition de chaînes crée les étiquettes.
Un graphique du mélange utilisera toutes les lignes ; un graphique
par composante utilisera la même table avec sa colonne `component`.

In [ ]:
def combine_components(values_a, values_b):
    """Return a long table with value and component columns."""
    # TODO: concatenate the two arrays and label every row A or B.
    return None

In [ ]:
combined = combine_components(np.array([1.0, 2.0]), np.array([3.0]))
if combined is None:
    print("À compléter : combine_components doit renvoyer une DataFrame.")
else:
    assert list(combined.columns) == ["value", "component"]
    assert combined["component"].tolist() == ["A", "A", "B"]
    assert combined["value"].tolist() == [1.0, 2.0, 3.0]
    display(combined)

## 4. Construire la classe progressivement

`GaussianMixture` est la **définition** d'un type. `model` sera une
**instance**. `__init__` enregistre ses paramètres dans `self` ;
`sample` utilise cet état, crée les données et les garde dans
`self.samples`. Les méthodes de graphique liront le même tableau.

Dans la cellule suivante, procédez dans cet ordre :

1. Ajoutez `self.std_a`, `self.mean_b`, `self.std_b`, `self.weight_a`.
2. Dans `sample`, calculez `n_a = int(n * self.weight_a)` et
   `n_b = n - n_a`. Si `n_a < 1` ou `n_b < 1`, signalez une erreur
   `ValueError` : chaque groupe doit contenir au moins une observation.
   Créez ensuite un générateur avec `seed`, appelez `draw_component`
   deux fois, puis `combine_components`. Affectez le résultat à
   `self.samples` et renvoyez-le.
3. Ajoutez les graphiques un par un. Relancez la cellule de classe
   et recréez `model` après chaque modification.

Valeurs à accepter : `n` est un entier au moins égal à 2 ; les
écarts-types sont strictement positifs et `0 < weight_a < 1`. Il faut
aussi que les deux effectifs calculés soient au moins égaux à 1.
Vérifiez ces conditions avant les tirages et signalez les valeurs
refusées avec `ValueError`.

Par exemple, `n=2` et `weight_a=0.1` donnent `n_a=0` et `n_b=2` :
ce cas doit être refusé. Avec `n=2` et `weight_a=0.5`, les deux
effectifs valent 1 : ce cas est accepté. Un seul point permet
de dessiner un groupe, mais ne permet pas d'en interpréter la dispersion.

In [ ]:
class GaussianMixture:
    """Two Gaussian components, one reproducible sample and three views."""

    def __init__(self, mean_a, std_a, mean_b, std_b, weight_a=0.6):
        self.mean_a = mean_a
        # TODO: store std_a, mean_b, std_b and weight_a on self.
        self.samples = None

    def sample(self, n, seed=404):
        """Draw n observations and keep them on this instance."""
        # TODO: use one generator, draw_component and combine_components.
        # n_a = int(n * self.weight_a); n_b = n - n_a
        return None

    def plot_components(self):
        """Overlay the two component histograms as densities."""
        # TODO: use px.histogram with color="component" and barmode="overlay".
        return None

    def plot_mixture(self):
        """Plot one histogram of all observations without component color."""
        # TODO: use px.histogram on self.samples.
        return None

    def plot_box(self):
        """Compare the two component samples with a box plot."""
        # TODO: use px.box with x="component" and y="value".
        return None

In [ ]:
model = GaussianMixture(100, 18, 106, 24, weight_a=0.6)
sample = model.sample(2000, seed=404)
if sample is None:
    print("À compléter : sample doit garder et renvoyer 2000 lignes.")
else:
    assert sample is model.samples
    assert sample.shape == (2000, 2)
    assert sample["component"].value_counts().to_dict() == {"A": 1200, "B": 800}
    display(sample.groupby("component")["value"].agg(["count", "mean", "std"]))
    print("Instance et échantillon validés.")

## 5. Trois méthodes, trois questions graphiques

Complétez les méthodes dans **la même cellule de classe** :

- `plot_components()` : `px.histogram(self.samples, x="value",
  color="component", histnorm="probability density",
  barmode="overlay", nbins=45, opacity=0.65)` ; comparer A et B.
- `plot_mixture()` : histogramme de `value` sans couleur de groupe ;
  observer ce qu'un jeu sans étiquette laisse voir.
- `plot_box()` : `px.box(self.samples, x="component", y="value")` ;
  comparer centres et dispersion.

**Choix de conception :** chaque méthode renvoie une figure. C'est
le notebook qui décide quand l'afficher ou l'enregistrer. Si
`sample` n'a pas encore été appelé, un `RuntimeError` explicite est
préférable à une erreur obscure dans Plotly.

In [ ]:
if sample is None:
    print("Terminez sample avant les graphiques.")
else:
    figures = {
        "components": model.plot_components(),
        "mixture": model.plot_mixture(),
        "box": model.plot_box(),
    }
    missing = [name for name, figure in figures.items() if figure is None]
    if missing:
        print("Méthodes à compléter :", ", ".join(missing))
    else:
        assert all(isinstance(figure, go.Figure) for figure in figures.values())
        for figure in figures.values():
            figure.show()

## 6. Deux objets, même code, résultats différents

Après avoir terminé la classe, changez `weight_a`, puis `std_b` dans
une seconde instance. Comparez les effectifs et les figures. Les
valeurs restent attachées à chaque instance : `first.samples` et
`second.samples` ne sont pas le même tableau.

In [ ]:
if sample is None:
    print("Terminez la classe avant de comparer deux instances.")
else:
    first = model
    second = GaussianMixture(100, 18, 106, 30, weight_a=0.3)
    second_sample = second.sample(2000, seed=404)
    assert first.samples is not second.samples
    print("A in first:", int((first.samples["component"] == "A").sum()))
    print("A in second:", int((second_sample["component"] == "A").sum()))

## 7. Déplacer les définitions dans un module

Dans l'éditeur, créez **vous-même** `my_gaussian_mixture.py` dans ce
dossier `continuation_22/`, à côté de ce notebook. Placez en haut `import numpy as np`,
`import pandas as pd` et `import plotly.express as px`. Copiez ensuite
vos deux fonctions et la classe, **sans** les données d'essai ni
les appels `figure.show()`. Enregistrez le fichier.

`import my_gaussian_mixture as gm` charge ce fichier dans un espace
de noms distinct. Le notebook conserve les expériences et les
appels ; le module contient les définitions réutilisables. Ne nommez
pas votre fichier `numpy.py`, `pandas.py` ou `plotly.py`.

Si le fichier manque, le contrôle indique simplement que cette
étape reste à faire. Si le fichier existe mais contient une erreur,
corrigez cette erreur : aucun module de secours ne la masque.

In [ ]:
personal_path = Path("my_gaussian_mixture.py")
print("Notebook directory:", Path.cwd())
print("Personal module:", personal_path.resolve())
module_build_ok = False
if not personal_path.is_file():
    print("À compléter : créer et enregistrer le module personnel.")
else:
    import my_gaussian_mixture as gm

    imported = gm.GaussianMixture(100, 18, 106, 24, weight_a=0.6)
    imported_sample = imported.sample(2000, seed=404)
    assert imported_sample.shape == (2000, 2)
    assert imported_sample["component"].value_counts().to_dict() == {"A": 1200, "B": 800}
    assert isinstance(imported.plot_components(), go.Figure)
    assert isinstance(imported.plot_mixture(), go.Figure)
    assert isinstance(imported.plot_box(), go.Figure)
    module_build_ok = True
    print("Imported from:", gm.__file__)
    print("Module validé :", module_build_ok)

## Vérifier et expliquer votre module

Enregistrez le `.py`, **redémarrez le noyau**, puis **exécutez tout**.
Un second `import` dans un noyau déjà actif peut conserver l'ancien
code ; un redémarrage vérifie aussi les dépendances entre cellules.
Le module est terminé seulement si `module_build_ok` vaut `True`.

Expliquez : quelle méthode crée `self.samples` ? Pourquoi les trois
figures renvoient-elles des objets plutôt que de faire `print` ?
Quel fichier modifier pour que les prochains imports changent ?

**Pour l'après-midi du 24 septembre :** suivez le
[livret DOCX](../../documents/2026-09-24_python_exercices.docx) :
exercice commun, puis choix A, B ou C. Le fichier
`05_Afternoon_exercises_FR.ipynb` conserve les quatre anciens exercices
du 22 septembre pour une reprise facultative ; ses lettres désignent
d'autres exercices.

Références : [fonctions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions),
[classes](https://docs.python.org/3/tutorial/classes.html),
[modules](https://docs.python.org/3/tutorial/modules.html),
[générateur NumPy](https://numpy.org/doc/stable/reference/random/generator.html),
[Plotly Express](https://plotly.com/python/plotly-express/).